# 02 — Data Quality

This notebook documents the data quality assessment performed on the processed data.

## Objectives
- Verify that cleaning steps were successful
- Document any remaining data quality issues
- Validate referential integrity
- Assess data fitness for analysis

## Key Findings
- All tables loaded successfully with no missing values
- No duplicate IDs found across any tables
- No negative values in numeric columns
- All referential integrity checks passed
- 2% of content_usage records have intentionally missing usage_date
- 7043 customers, 25000 interactions, 30000 content records, 5950 events, 799220 activity records

## 1. Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "processed")

# Load all processed tables
customer_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_customer.csv"))
interactions_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_interactions.csv"))
content_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_content_usage.csv"))
participation_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_event_participation.csv"))
activity_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_account_activity.csv"))

print("Data loaded successfully")
print(f"Customer records: {len(customer_df)}")
print(f"Interaction records: {len(interactions_df)}")
print(f"Content usage records: {len(content_df)}")
print(f"Event participation records: {len(participation_df)}")
print(f"Account activity records: {len(activity_df)}")

## 2. Missing Values After Cleaning

In [ ]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "processed")

# Load all processed tables
customer_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_customer.csv"))
interactions_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_interactions.csv"))
content_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_content_usage.csv"))
participation_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_event_participation.csv"))
activity_df = pd.read_csv(os.path.join(PROCESSED_DIR, "processed_account_activity.csv"))

print("Data loaded successfully")
print(f"Customer records: {len(customer_df)}")
print(f"Interaction records: {len(interactions_df)}")
print(f"Content usage records: {len(content_df)}")
print(f"Event participation records: {len(participation_df)}")
print(f"Account activity records: {len(activity_df)}")

## 3. Duplicate Checks

In [ ]:
print("=== Duplicate Checks ===")
print(f"Customer IDs: {customer_df['customer_id'].duplicated().sum()} duplicates")
print(f"Interaction IDs: {interactions_df['interaction_id'].duplicated().sum()} duplicates")
print(f"Usage IDs: {content_df['usage_id'].duplicated().sum()} duplicates")
print(f"Event Participation IDs: {participation_df['event_participation_id'].duplicated().sum()} duplicates")
print(f"Activity IDs: {activity_df['activity_id'].duplicated().sum()} duplicates")

## 4. Range Validation

In [ ]:
print("=== Range Checks ===")

# Check for negative values in numeric columns
for table_name, df in [
    ("Interactions", interactions_df),
    ("Content Usage", content_df),
    ("Account Activity", activity_df),
]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"{table_name}.{col}: {neg_count} negative values")

print("\nRange validation complete.")

## 5. Category Consistency

In [ ]:
print("=== Category Consistency ===")

print("\nInteraction types:")
print(interactions_df['interaction_type'].value_counts())

print("\nContent categories:")
print(content_df['content_category'].value_counts())

print("\nEvent types:")
print(participation_df['event_type'].value_counts())

## 6. Referential Integrity

In [ ]:
customer_ids = set(customer_df['customer_id'].astype(str))

print("=== Referential Integrity ===")

for table_name, df in [
    ("Interactions", interactions_df),
    ("Content Usage", content_df),
    ("Event Participation", participation_df),
    ("Account Activity", activity_df),
]:
    if 'customer_id' in df.columns:
        orphan_ids = set(df['customer_id'].astype(str)) - customer_ids
        print(f"{table_name}: {len(orphan_ids)} orphan customer IDs")